In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

# Models:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.classification import MultilayerPerceptronClassifier



In [0]:

auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="recallByLabel"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="fMeasureByLabel"
)


In [0]:

df = spark.read.table("teams.sensorx.df_sx_800_with_delta")
df = df.drop("serialNumber", "payload_fault_faultName", "GeneratorType")

sensor_cols = [c for c in df.columns if c not in {"properties_deviceId", "timeStamp", "payload_fault_state"}]

df =df.na.drop(subset=sensor_cols)

# --- 2) One-Hot Encode properties_deviceId ---
indexer = StringIndexer(inputCol="properties_deviceId", outputCol="deviceId_index")
df_indexed = indexer.fit(df).transform(df)

encoder = OneHotEncoder(inputCol="deviceId_index", outputCol="deviceId_ohe")
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

# Horizon
H_days = 15  # horizon in days
H_seconds = H_days * 86400  # convert to seconds

w_horizon = (
    Window.partitionBy("properties_deviceId")
    .orderBy(F.col("timeStamp").cast("long"))   # order by epoch seconds
    .rangeBetween(0, H_seconds)                 # look ahead H_seconds
)

# count number of rows with 0 and 1 in fault_state
df_encoded = df_encoded.withColumn(
    "failure_horizon",
    F.max(F.col("payload_fault_state").cast("int")).over(w_horizon)
)

n_lags = 20
w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

for L in range(1, n_lags + 1):
    for col_name in sensor_cols:
        df_encoded = df_encoded.withColumn(f"{col_name}{L}", F.lag(col_name, L).over(w))

lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

df_clean = df_encoded.na.drop(subset=lag_cols)

# Assemble features
feature_input_cols = sensor_cols + lag_cols
assembler = VectorAssembler(
    inputCols=feature_input_cols, 
    outputCol="features")
df_features = assembler.transform(df_clean) \
    .select("timeStamp", "features", F.col("failure_horizon").alias("label"))

# Chronological test train split 80/20
cutoff_epoch = df_features.selectExpr(
    "percentile_approx(CAST(timeStamp AS BIGINT), 0.8)"
).first()[0]

train = df_features.filter(F.col("timeStamp").cast("bigint") <= cutoff_epoch).cache()
test  = df_features.filter(F.col("timeStamp").cast("bigint") >  cutoff_epoch).cache()

In [0]:
spark.sql("DROP TABLE IF EXISTS teams.sensorx.train_features_delta")
spark.sql("DROP TABLE IF EXISTS teams.sensorx.test_features_delta")

train.write.saveAsTable("teams.sensorx.train_features_delta")
test.write.saveAsTable("teams.sensorx.test_features_delta")

In [0]:
train=spark.read.table("teams.sensorx.train_features_delta")
test=spark.read.table("teams.sensorx.test_features_delta")

# Class weighting (give minority class higher weight)
label_counts = train.groupBy("label").count().collect()
total = sum(row["count"] for row in label_counts)
weight_map = {row["label"]: total / (2.0 * row["count"]) for row in label_counts}
#print(f"Class weights: {weight_map}")

# Add weight column to train and test
weight_expr = F.when(F.col("label") == 1, weight_map[1]).otherwise(weight_map[0])
train_w = train.withColumn("weight", weight_expr)
test_w = test.withColumn("weight", weight_expr)

In [0]:

# Raw fault_state distribution
print("Raw payload_fault_state distribution:")
df.groupBy("payload_fault_state").count().orderBy("payload_fault_state").show()

# After failure horizon
print(f"Failure horizon (H={H_days} days) distribution:")
df_clean.groupBy("failure_horizon").count().orderBy("failure_horizon").show()

In [0]:
# --- RandomForest with class weighting ---
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=100,
    seed=42
)
RFmodel = rf.fit(train_w)
RFpredictions = RFmodel.transform(test_w)

RFtrainingSummary = RFmodel.summary
RF_auc = auc_eval.evaluate(RFpredictions)

# Recall (per class)
RF_recall_0 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 0.0})
RF_recall_1 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
RF_f1_0 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 0.0})
RF_f1_1 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nRandom Forest with {n_lags} lags (class-weighted)")
print(f"AUC: {RF_auc:.4f}")
print(f"Recall (label=0): {RF_recall_0:.4f}")
print(f"Recall (label=1): {RF_recall_1:.4f}")
print(f"F1     (label=0): {RF_f1_0:.4f}")
print(f"F1     (label=1): {RF_f1_1:.4f}")

# Confusion matrix
RFpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

Gradient Boosted trees with lag and horizon:

skoða weights, see feature importance for the trees


In [0]:
# GBT training
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=30,
    seed=42
)

GBTmodel = gbt.fit(train_w)
gbtpredictions = GBTmodel.transform(test_w)

# --- Training accuracy ---
gbt_train_preds = GBTmodel.transform(train_w)
GBT_train_auc = auc_eval.evaluate(gbt_train_preds)

# --- Test evaluation ---
GBT_auc = auc_eval.evaluate(gbtpredictions)

# Recall (per class)
GBT_recall_0 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 0.0})
GBT_recall_1 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
GBT_f1_0 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 0.0})
GBT_f1_1 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nGBT with {n_lags} lags (class-weighted)")
print(f"Training AUC: {GBT_train_auc:.4f}")
print(f"Test AUC:     {GBT_auc:.4f}")
print(f"Recall (label=0): {GBT_recall_0:.4f}")
print(f"Recall (label=1): {GBT_recall_1:.4f}")
print(f"F1     (label=0): {GBT_f1_0:.4f}")
print(f"F1     (label=1): {GBT_f1_1:.4f}")

# Confusion matrix
gbtpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

# --- Feature importance ---
ohe_size = GBTmodel.numFeatures - len(sensor_cols) - len(lag_cols)
ohe_names = [f"deviceId_ohe_{i}" for i in range(ohe_size)]
all_feature_names = sensor_cols + ohe_names + lag_cols

importances = GBTmodel.featureImportances.toArray().tolist()
feat_imp_df = spark.createDataFrame(
    zip(all_feature_names, importances), ["feature", "importance"]
).orderBy(F.desc("importance"))

print(f"\nTop 20 features by importance:")
display(feat_imp_df.limit(20))

In [0]:
import mlflow
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

CATALOG = "teams"
SCHEMA = "sensorx"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.gbt_fault_prediction"

with mlflow.start_run(run_name="GBT_fault_prediction") as run:
    # Log parameters
    mlflow.log_param("model_type", "GBTClassifier")
    mlflow.log_param("maxIter", 30)
    mlflow.log_param("n_lags", n_lags)
    mlflow.log_param("H_days", H_days)
    mlflow.log_param("sensor_cols", sensor_cols)

    # Log metrics
    mlflow.log_metric("AUC", GBT_auc)
    mlflow.log_metric("Recall_label_0", GBT_recall_0)
    mlflow.log_metric("Recall_label_1", GBT_recall_1)
    mlflow.log_metric("F1_label_0", GBT_f1_0)
    mlflow.log_metric("F1_label_1", GBT_f1_1)

    # Infer signature from a small sample
    sample_input = test_gbt.select("features").limit(5)
    sample_output = GBTmodel.transform(sample_input).select("prediction")
    signature = infer_signature(
        sample_input.toPandas(),
        sample_output.toPandas()
    )

    # Log and register the Spark ML model to Unity Catalog
    model_info = mlflow.spark.log_model(
        GBTmodel,
        artifact_path="gbt_model",
        signature=signature,
        input_example=sample_input.limit(1).toPandas(),
        registered_model_name=MODEL_NAME,
    )

    print(f"Model registered to: {MODEL_NAME}")
    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")
    print(f"AUC: {GBT_auc:.4f}")
    print(f"Recall (label=1): {GBT_recall_1:.4f}")
    print(f"F1 (label=1): {GBT_f1_1:.4f}")

In [0]:
from pyspark.ml.functions import vector_to_array

# Run predictions on full test set
all_predictions = GBTmodel.transform(test_gbt)

# Join back with original data to get deviceId and actual fault_state
original_cols = df_encoded.select(
    "timeStamp", "properties_deviceId", 
    F.col("payload_fault_state").cast("int").alias("actual_fault_state")
)

timeline = (
    all_predictions
    .join(original_cols, on="timeStamp", how="inner")
    .select(
        "timeStamp",
        "properties_deviceId",
        F.col("actual_fault_state").alias("fault_now"),
        F.col("label").alias("fault_within_15d"),
        F.col("prediction").cast("int").alias("predicted_fault_within_15d"),
        vector_to_array("probability")[1].alias("prob_fault")
    )
    .orderBy("properties_deviceId", "timeStamp")
)

# Pick a device that actually has a fault transition (fault_now goes 0 → 1)
device_with_fault = (
    timeline
    .filter(F.col("fault_now") == 1)
    .select("properties_deviceId")
    .first()
)

if device_with_fault:
    device_id = device_with_fault["properties_deviceId"]
    print(f"Showing timeline for device: {device_id}")
    print(f"fault_now = actual payload_fault_state at this moment")
    print(f"fault_within_15d = label (1 if a fault occurs within 15 days)")
    print(f"predicted_fault_within_15d = model prediction\n")
    
    device_timeline = timeline.filter(F.col("properties_deviceId") == device_id)
    display(device_timeline)
else:
    print("No device with fault transition found in test set")
    display(timeline.limit(100))

In [0]:
# Normalize features for MLP (fit on train only to avoid data leakage)
scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(train)

train_norm = scaler_model.transform(train).drop("features").withColumnRenamed("features_scaled", "features")
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

Neural Network testing

-- check tuning learning rate, where to do that ? (mögulega step size) 

In [0]:

# Input layer = 161 features, one hidden layer, output = 2 classes (binary)
layers = [126,16,16,2]

trainer = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    layers=layers,
    blockSize=128,
    seed=42,
    stepSize=0.001  #try different values of this - too low will underfit, too high might overfit training set
)

# Use normalized data (MLP benefits from normalization, but no weightCol support)
MLPmodel = trainer.fit(train_norm)
MLPpredictions = MLPmodel.transform(test_norm)

# --- Evaluate (same format as RF / GBT) ---
MLP_training_auc = auc_eval.evaluate(MLPmodel.transform(train_norm))
MLP_auc = auc_eval.evaluate(MLPpredictions)

MLP_recall_0 = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: 0.0})
MLP_recall_1 = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: 1.0})

MLP_f1_0 = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: 0.0})
MLP_f1_1 = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nMLP Neural Network with {n_lags} lags (normalized, no class weighting)")
print(f"Layers: {layers}")
print(f"AUC: {MLP_auc:.4f}")
print(f"Training AUC: {MLP_training_auc:.4f}")
print(f"Recall (label=0): {MLP_recall_0:.4f}")
print(f"Recall (label=1): {MLP_recall_1:.4f}")
print(f"F1     (label=0): {MLP_f1_0:.4f}")
print(f"F1     (label=1): {MLP_f1_1:.4f}")

# Confusion matrix
MLPpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()